# Supervised Fine-Tuning

Supervised Fine-Tuning (SFT) is the foundational stage in the Reinforcement Learning from Human Feedback (RLHF) pipeline. It adapts a pretrained language model to follow human instructions by training it on labeled input–output pairs. In most modern alignment pipelines, SFT serves as the initial policy from which preference optimization or reinforcement learning methods are applied.

###Role of SFT in the RLHF Pipeline

The RLHF pipeline typically consists of three stages:

1) Supervised Fine-Tuning (SFT)
The pretrained model is fine-tuned on curated datasets of human-written responses. This teaches the model basic instruction-following behavior and linguistic style.

2) Preference Learning
Human annotators rank multiple model-generated responses. These rankings are used to learn preferences or optimize policies directly (e.g., PPO, DPO).

3) Policy Optimization
The model is further optimized to align with human preferences while maintaining proximity to the SFT policy.

SFT is crucial because preference optimization assumes a reasonable starting policy. Without SFT, reinforcement or preference-based methods become unstable and inefficient.

##Model Choice: GPT-2

We use GPT-2, a decoder-only Transformer model, along with its corresponding tokenizer from the Hugging Face transformers library.

###Why GPT-2?

-Lightweight and computationally efficient

-Well-understood architecture

-Suitable for experimentation on limited hardware

-Demonstrates SFT behavior clearly without heavy infrastructure

GPT-2 is pretrained on large-scale internet text and learns general language modeling capabilities. SFT adapts these capabilities to a task-specific distribution.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

## Testing the Tokenizer

### Tokenize a batch

In [ ]:
texts = ['I am a human', 'I have a dog', 'I also have a cat']
tokens_obj = tokenizer(texts)

In [ ]:
for tokens in tokens_obj['input_ids']:
    print(tokenizer.decode(tokens))

In this notebook, we perform SFT using the Stanford Sentiment Treebank v2 (SST-2) dataset, accessed via the Hugging Face datasets library as stanfordnlp/sst2.

##Dataset Characteristics

Task: Binary sentiment classification

Inputs: Natural language sentences

Labels:

0 → Negative sentiment

1 → Positive sentiment

Although SST-2 is traditionally a classification dataset, it can be reframed as an instruction-following task by converting labels into natural language responses. For example:

In [ ]:
%pip install datasets==3.5.0

### Loading a dataset

In [ ]:
from datasets import load_dataset
dataset_name = 'sst2'
ds = load_dataset(dataset_name)

In [ ]:
ds_train, ds_val = ds['train'], ds['validation']
ds_train

## Tokenizing a Dataset

In [ ]:
def tokenize(batch):
    return tokenizer(batch['sentence'])

map_kwargs = {
    'batched': True,
    'batch_size': 512,
    'remove_columns': ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

In [ ]:
tokenized_dataset_train[0]

In [ ]:
tokenized_dataset_train[5:10]

### Decoding from the dataset

In [ ]:
for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i+1}: {tokenizer.decode(seq)}')

### Filter out tweets shorter than 5 tokens

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

In [ ]:
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 5)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 5)

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

## Preparing a dataloader

### Set PyTorch format

In [ ]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [ ]:
tokenized_dataset_train[0]

In [ ]:
tokenized_dataset_train[:5]

### Padding

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

### Collation with Padding

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # labels

dataloader_params = {
    'batch_size': 32,
    'collate_fn': data_collator
}

train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [ ]:
len(train_dataloader)

In [ ]:
1544 * 32

In [ ]:
batch = next(iter(train_dataloader))
print(batch.keys())

## Supervised Fine-tuning (SFT)

In [ ]:
import torch
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 1

### Training loop

In [ ]:
def validate(epoch):
    model.eval()
    total_loss = 0.0
    for i, batch in enumerate(val_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
    print(f'val_loss at {epoch} epoch:', total_loss / len(val_dataloader))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
validate(0)
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss
        print(f'Loss: {loss.item()}')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    validate(epoch+1)

### Save the model

In [ ]:
model.save_pretrained('./sft_model_epoch_1')

In [ ]:
model.from_pretrained('./sft_model_epoch_1')